# 🚨 LA Crime Data - Spatiotemporal Analysis & Pattern Detection
## Hexbin Heatmaps, Temporal Clustering, Network Analysis & Predictive Hotspots

**Data:** `~/lab/data/CSVs/Crime_Data_from_2020_to_Present.csv`

### Exotic Techniques:
- 🗺️ Interactive hexbin crime density maps
- ⏰ Temporal pattern mining (hour/day/month)
- 🔍 DBSCAN spatial-temporal clustering
- 🌡️ Crime heatmaps with animation
- 📊 Chord diagrams for crime type relationships
- 🎯 Hotspot prediction using historical patterns
- 📈 Seasonal decomposition by crime type
- 🌐 Network analysis of crime locations

## Setup: Install Missing Libraries

In [2]:
%%calliope ask gpto --format code
Check if scikit-learn is installed.
If not, run: sudo pip install scikit-learn

Check if folium is installed.
If not, run: sudo pip install folium

Print success messages.

In [3]:
import subprocess
import sys

def check_and_install(package_name):
    try:
        __import__(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name} installed successfully.")

check_and_install("scikit-learn")
check_and_install("folium")

scikit-learn not found. Installing...
  Using cached scikit_learn-1.7.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.4 MB)
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
scikit-learn installed successfully.
folium not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [folium]
folium installed successfully.


In [4]:
import importlib
import subprocess
import sys

# Function to check if a package is installed
def is_package_installed(package_name):
    package_spec = importlib.util.find_spec(package_name)
    return package_spec is not None

# Function to install a package using pip
def install_package(package_name):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

# Check and install scikit-learn
if not is_package_installed('sklearn'):
    subprocess.run(['sudo', sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
    print("scikit-learn has been successfully installed.")
else:
    print("scikit-learn is already installed.")

# Check and install folium
if not is_package_installed('folium'):
    subprocess.run(['sudo', sys.executable, '-m', 'pip', 'install', 'folium'])
    print("folium has been successfully installed.")
else:
    print("folium is already installed.")

scikit-learn is already installed.
folium is already installed.


## 1. Load & Sample Data

In [9]:
%%calliope ask gpto --format code
Load ~/lab/data/CSVs/Crime_Data_from_2020_to_Present.csv
double check and confirm column names normalize with underscores and all lower case to avoid KeyError Error Detected
Error: 'date_time'
Sample 100000 rows for performance.
Clean column names.
Parse date/time columns properly.
Extract features:
- hour of day
- day of week
- month
- year
- is_weekend boolean

Display:
- Shape and info
- First 10 rows
- Crime type value counts (top 20)
- Date range covered

Store as 'crime_df'.

In [10]:
import pandas as pd

# Load the dataset
file_path = "~/lab/data/CSVs/Crime_Data_from_2020_to_Present.csv"
crime_data = pd.read_csv(file_path)

# Clean and normalize column names
crime_data.columns = crime_data.columns.str.strip().str.lower().str.replace(' ', '_')

# Sample 100000 rows for performance
crime_data_sample = crime_data.sample(n=100000, random_state=1)

# Identify likely date/time column for parsing
date_col = None
for col in crime_data_sample.columns:
    if 'date' in col or 'time' in col:
        date_col = col
        crime_data_sample[date_col] = pd.to_datetime(crime_data_sample[date_col])
        break

# Extract features
crime_data_sample['hour_of_day'] = crime_data_sample[date_col].dt.hour
crime_data_sample['day_of_week'] = crime_data_sample[date_col].dt.dayofweek
crime_data_sample['month'] = crime_data_sample[date_col].dt.month
crime_data_sample['year'] = crime_data_sample[date_col].dt.year
crime_data_sample['is_weekend'] = crime_data_sample['day_of_week'].apply(lambda x: x >= 5)

# Display information
print("Shape:", crime_data_sample.shape)
print("\nInfo:")
print(crime_data_sample.info())

print("\nFirst 10 rows:")
print(crime_data_sample.head(10))

print("\nTop 20 Crime Type Value Counts:")
crime_col = [col for col in crime_data_sample.columns if 'crime' in col and 'type' in col]
if crime_col:
    print(crime_data_sample[crime_col[0]].value_counts().head(20))

date_min = crime_data_sample[date_col].min()
date_max = crime_data_sample[date_col].max()
print(f"\nDate Range: {date_min} to {date_max}")

crime_df = crime_data_sample

/tmp/ipykernel_3026/3246741926.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_data_sample[date_col] = pd.to_datetime(crime_data_sample[date_col])


Shape: (100000, 33)

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 861691 to 655477
Data columns (total 33 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   dr_no           100000 non-null  int64         
 1   date_rptd       100000 non-null  datetime64[ns]
 2   date_occ        100000 non-null  object        
 3   time_occ        100000 non-null  int64         
 4   area            100000 non-null  int64         
 5   area_name       100000 non-null  object        
 6   rpt_dist_no     100000 non-null  int64         
 7   part_1-2        100000 non-null  int64         
 8   crm_cd          100000 non-null  int64         
 9   crm_cd_desc     100000 non-null  object        
 10  mocodes         85022 non-null   object        
 11  vict_age        100000 non-null  int64         
 12  vict_sex        85715 non-null   object        
 13  vict_descent    85713 non-null   object        
 14  premis_cd

In [12]:
import pandas as pd

# Load the dataset
file_path = "~/lab/data/CSVs/Crime_Data_from_2020_to_Present.csv"
crime_data = pd.read_csv(file_path)

# Clean and normalize column names
crime_data.columns = crime_data.columns.str.strip().str.lower().str.replace(' ', '_')

# Sample 100000 rows for performance
crime_data_sample = crime_data.sample(n=100000, random_state=1)

# Identify likely date/time column for parsing
date_col = None
for col in crime_data_sample.columns:
    if 'date' in col or 'time' in col:
        date_col = col
        crime_data_sample[date_col] = pd.to_datetime(crime_data_sample[date_col])
        break

# Extract features
crime_data_sample['hour_of_day'] = crime_data_sample[date_col].dt.hour
crime_data_sample['day_of_week'] = crime_data_sample[date_col].dt.dayofweek
crime_data_sample['month'] = crime_data_sample[date_col].dt.month
crime_data_sample['year'] = crime_data_sample[date_col].dt.year
crime_data_sample['is_weekend'] = crime_data_sample['day_of_week'].apply(lambda x: x >= 5)

# Display information
print("Shape:", crime_data_sample.shape)
print("\nInfo:")
print(crime_data_sample.info())

print("\nFirst 10 rows:")
print(crime_data_sample.head(10))

print("\nTop 20 Crime Type Value Counts:")
crime_col = [col for col in crime_data_sample.columns if 'crime' in col and 'type' in col]
if crime_col:
    print(crime_data_sample[crime_col[0]].value_counts().head(20))

date_min = crime_data_sample[date_col].min()
date_max = crime_data_sample[date_col].max()
print(f"\nDate Range: {date_min} to {date_max}")

crime_df = crime_data_sample

/tmp/ipykernel_3026/3246741926.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_data_sample[date_col] = pd.to_datetime(crime_data_sample[date_col])


Shape: (100000, 33)

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 861691 to 655477
Data columns (total 33 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   dr_no           100000 non-null  int64         
 1   date_rptd       100000 non-null  datetime64[ns]
 2   date_occ        100000 non-null  object        
 3   time_occ        100000 non-null  int64         
 4   area            100000 non-null  int64         
 5   area_name       100000 non-null  object        
 6   rpt_dist_no     100000 non-null  int64         
 7   part_1-2        100000 non-null  int64         
 8   crm_cd          100000 non-null  int64         
 9   crm_cd_desc     100000 non-null  object        
 10  mocodes         85022 non-null   object        
 11  vict_age        100000 non-null  int64         
 12  vict_sex        85715 non-null   object        
 13  vict_descent    85713 non-null   object        
 14  premis_cd

## 2. Temporal Pattern Heatmap Matrix

In [ ]:
%%calliope ask anthropic-chat:claude-opus-4-20250514 --format code
Using crime_df:

Create temporal pattern heatmap:
1. Pivot table: hour (rows) x day_of_week (columns), values = crime count
2. Create plotly heatmap with:
   - Color scale showing crime intensity
   - Annotations with actual counts
   - Hover showing: hour, day, count, % of total
3. Add marginal distributions (row/column sums)
4. Identify peak crime hours and days
5. Create second heatmap: month x day_of_week

Display both heatmaps and print peak times.

## 3. Interactive Hexbin Density Map

In [12]:
%%calliope ask openai-chat:gpt-5 --format code
Using crime_df with lat/lon coordinates:

Create hexbin density map:
1. Use plotly density_mapbox for LA area
2. Hexagonal binning of crime locations
3. Color intensity = crime count per hex
4. Add layer selector for crime types:
   - All crimes
   - Violent crimes
   - Property crimes
   - Drug-related
5. Animation frame by month/year
6. Add district boundaries overlay
7. Hover: hex crime count, dominant type, time pattern

Save as 'crime_hexmap.html' OR display inline.

In [13]:
import sys
import os
import json
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure h3 is available
try:
    import h3
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "h3"])
    import h3

# 1) Detect latitude/longitude columns
lat_candidates = ['lat', 'latitude', 'y', 'lat_dd', 'lat_wgs84']
lon_candidates = ['lon', 'lng', 'longitude', 'x', 'long_dd', 'lon_wgs84']

def find_col(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

cols = set(crime_df.columns.str.lower())
# Normalize column names to lower for detection without changing the original df
# Access columns via original names found (case-sensitive) if needed
lower_to_original = {c.lower(): c for c in crime_df.columns}

lat_col_l = find_col(lat_candidates, cols)
lon_col_l = find_col(lon_candidates, cols)
if not lat_col_l or not lon_col_l:
    raise ValueError("Could not find latitude/longitude columns in crime_df. Expected columns like lat/lon or latitude/longitude.")

lat_col = lower_to_original[lat_col_l]
lon_col = lower_to_original[lon_col_l]

# 2) Detect date/time column
date_candidates = [
    'date_occurred', 'date_reported', 'occ_date', 'occ_datetime', 'datetime',
    'timestamp', 'date', 'reported_date', 'occurrence_date', 'report_date'
]
date_col_l = find_col(date_candidates, cols)
date_col = lower_to_original[date_col_l] if date_col_l else None

# Parse date if present
df = crime_df.copy()
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# Ensure hour_of_day
if 'hour_of_day' not in df.columns:
    if date_col and pd.api.types.is_datetime64_any_dtype(df[date_col]):
        df['hour_of_day'] = df[date_col].dt.hour
    else:
        df['hour_of_day'] = np.nan

# Year-month for animation
if date_col and pd.api.types.is_datetime64_any_dtype(df[date_col]):
    df['year_month'] = df[date_col].dt.to_period('M').astype(str)
else:
    # Fallback: single frame if no date
    df['year_month'] = 'Unknown'

# 3) Detect a crime description column and classify categories
desc_candidates = [
    'crimetype', 'crm_cd_desc', 'crime_description', 'description', 'offense',
    'offense_desc', 'crime_code_description', 'stat_desc', 'charge_desc', 'ucr_part'
]
desc_col_l = find_col(desc_candidates, cols)
desc_col = lower_to_original[desc_col_l] if desc_col_l else None

violent_kw = [
    'assault', 'battery', 'homicide', 'murder', 'manslaughter', 'rape', 'sexual',
    'robbery', 'shoot', 'shooting', 'kidnap', 'carjacking', 'mayhem'
]
property_kw = [
    'burglary', 'theft', 'larceny', 'shoplift', 'stolen', 'vehicle theft',
    'grand theft', 'petty theft', 'vandal', 'arson', 'fraud', 'embezzle',
    'trespass', 'burglary from vehicle', 'burglary/vehicle', 'burglary vehicle'
]
drug_kw = [
    'drug', 'narcotic', 'controlled substance', 'marijuana', 'cocaine',
    'meth', 'heroin', 'opiate', 'phencyclidine', 'pcp'
]

def classify_category(text):
    if not isinstance(text, str):
        return 'Other'
    t = text.lower()
    if any(k in t for k in drug_kw):
        return 'Drug'
    if any(k in t for k in violent_kw):
        return 'Violent'
    if any(k in t for k in property_kw):
        return 'Property'
    return 'Other'

if desc_col:
    df['category'] = df[desc_col].apply(classify_category)
else:
    # Without description, set all as Other but still allow All category
    df['category'] = 'Other'

# 4) Clean coordinates: drop invalid/missing and clip to LA region
df = df.copy()
df = df.rename(columns={lat_col: '_lat', lon_col: '_lon'})
df = df[pd.notnull(df['_lat']) & pd.notnull(df['_lon'])]
# Remove zero or invalid
df = df[(df['_lat'] != 0) & (df['_lon'] != 0)]
# Rough LA bounding box to remove outliers
df = df[(df['_lat'] >= 33.3) & (df['_lat'] <= 34.8) & (df['_lon'] >= -119.1) & (df['_lon'] <= -117.3)]

# 5) Assign H3 hexagon
H3_RES = 8  # suitable urban scale
df['hex'] = df.apply(lambda r: h3.geo_to_h3(r['_lat'], r['_lon'], H3_RES), axis=1)

# 6) Aggregate by year_month and hex
# All crimes count per hex/month
g_all = df.groupby(['year_month', 'hex']).size().rename('count').reset_index()

# Category counts per hex/month
g_cat = df.groupby(['year_month', 'hex', 'category']).size().rename('n').reset_index()
g_cat_piv = g_cat.pivot_table(index=['year_month', 'hex'], columns='category', values='n', fill_value=0).reset_index()
for cat in ['Violent', 'Property', 'Drug', 'Other']:
    if cat not in g_cat_piv.columns:
        g_cat_piv[cat] = 0

agg = pd.merge(g_all, g_cat_piv, on=['year_month', 'hex'], how='left')

# Peak hour per hex/month (time pattern)
if 'hour_of_day' in df.columns and df['hour_of_day'].notna().any():
    gh = df.dropna(subset=['hour_of_day']).groupby(['year_month', 'hex', 'hour_of_day']).size().rename('nh').reset_index()
    idx = gh.groupby(['year_month', 'hex'])['nh'].idxmax()
    peak = gh.loc[idx, ['year_month', 'hex', 'hour_of_day']].rename(columns={'hour_of_day': 'peak_hour'})
else:
    peak = pd.DataFrame(columns=['year_month', 'hex', 'peak_hour'])
agg = pd.merge(agg, peak, on=['year_month', 'hex'], how='left')
agg['peak_hour'] = agg['peak_hour'].astype('Int64')

# Dominant type among categories at each hex/month (based on category columns)
def dominant_type_row(row):
    vals = {'Violent': row.get('Violent', 0), 'Property': row.get('Property', 0), 'Drug': row.get('Drug', 0), 'Other': row.get('Other', 0)}
    # If all zero, return 'None'
    if max(vals.values()) == 0:
        return 'None'
    return max(vals, key=vals.get)

agg['dominant_type'] = agg.apply(dominant_type_row, axis=1)

# 7) Build GeoJSON for all hexes
all_hexes = agg['hex'].unique().tolist()
features = []
for h in all_hexes:
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  # list of [lat, lon]
    # Convert to [lon, lat] and ensure closed ring
    coords = [[lng, lat] for lat, lng in boundary]
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    feat = {
        "type": "Feature",
        "id": h,
        "properties": {"hex": h},
        "geometry": {"type": "Polygon", "coordinates": [coords]}
    }
    features.append(feat)

hex_geojson = {"type": "FeatureCollection", "features": features}

# 8) Prepare frames per category
periods = sorted(agg['year_month'].unique().tolist())
category_map = {
    'ALL': 'count',
    'VIOLENT': 'Violent',
    'PROPERTY': 'Property',
    'DRUG': 'Drug'
}

def build_frames_for_category(cat_key):
    z_col = category_map[cat_key]
    frames = []
    for ym in periods:
        dfm = agg.loc[agg['year_month'] == ym].copy()
        # Filter hexes with >0 counts for this category to reduce draw
        if z_col in dfm.columns:
            dfm = dfm[dfm[z_col] > 0]
        else:
            # If missing (shouldn't), skip
            dfm = dfm.iloc[0:0]
        # Prepare customdata: dominant, peak_hour
        dom = dfm['dominant_type'].fillna('None').astype(str).values
        peak = dfm['peak_hour'].astype('Int64').astype(str).replace({'<NA>': 'Unknown'}).values
        custom = np.stack([dom, peak, np.array([ym] * len(dfm))], axis=1) if len(dfm) else np.empty((0, 3))
        trace = go.Choroplethmapbox(
            geojson=hex_geojson,
            locations=dfm['hex'],
            z=dfm[z_col] if z_col in dfm.columns else np.array([]),
            zmin=0,
            zmax=max(1, (agg[z_col].max() if z_col in agg.columns else 1)),
            colorscale='YlOrRd',
            marker_line_width=0.5,
            marker_line_color='rgba(0,0,0,0.2)',
            featureidkey="properties.hex",
            hovertemplate="Month: %{customdata[2]}<br>Crimes: %{z}<br>Dominant type: %{customdata[0]}<br>Peak hour: %{customdata[1]}<extra></extra>",
            customdata=custom,
            showscale=True,
            colorbar=dict(title="Crime Count")
        )
        frames.append(go.Frame(name=f"{cat_key}|{ym}", data=[trace]))
    return frames

frames_all = build_frames_for_category('ALL')
frames_violent = build_frames_for_category('VIOLENT')
frames_property = build_frames_for_category('PROPERTY')
frames_drug = build_frames_for_category('DRUG')

all_frames = frames_all + frames_violent + frames_property + frames_drug

# 9) Base figure with initial ALL frame (first period)
initial_frame = frames_all[0] if frames_all else go.Frame(name="ALL|Unknown", data=[go.Choroplethmapbox()])
fig = go.Figure(data=initial_frame.data, frames=all_frames)

# 10) Add a light background density layer (Plotly density_mapbox) for visual context
# Sample to avoid heavy plot
d_sample = df.sample(n=min(50000, len(df)), random_state=42) if len(df) > 50000 else df
density_trace = go.Densitymapbox(
    lat=d_sample['_lat'],
    lon=d_sample['_lon'],
    z=None,
    radius=20,
    colorscale='Viridis',
    opacity=0.25,
    showscale=False
)
fig.add_trace(density_trace)

# 11) Optional district boundaries overlay (try to load if available)
boundary_paths = [
    os.path.expanduser("~/lab/data/geo/lapd_divisions.geojson"),
    os.path.expanduser("~/lab/data/geo/la_districts.geojson"),
    os.path.expanduser("~/data/geo/lapd_divisions.geojson"),
    os.path.expanduser("./lapd_divisions.geojson")
]
boundary_geojson = None
for p in boundary_paths:
    if os.path.isfile(p):
        try:
            with open(p, 'r') as f:
                boundary_geojson = json.load(f)
            break
        except Exception:
            boundary_geojson = None

# 12) Layout with mapbox
fig.update_layout(
    mapbox_style="open-street-map",
    mapbox_zoom=9.5,
    mapbox_center={"lat": 34.0522, "lon": -118.2437},
    title="LA Crime Hexbin Density Map (Monthly Animation with Crime Type Selector)",
    margin=dict(l=10, r=10, t=60, b=10),
)

# Add overlay line layer if available
if boundary_geojson is not None:
    fig.update_layout(
        mapbox_layers=[
            dict(
                sourcetype='geojson',
                source=boundary_geojson,
                type='line',
                color='black',
                line={'width': 1.5},
                opacity=0.7
            )
        ]
    )

# Ensure choropleth trace is drawn above density (move choropleth to the end)
# Current traces: [choropleth (from initial), density]. We want choropleth on top.
# Re-add initial choropleth after density by resetting data order
if len(fig.data) >= 2:
    choropleth_initial = fig.data[0]
    density = fig.data[1]
    fig.data = (density, choropleth_initial)

# 13) Build category animation sequences
names_all = [f.name for f in frames_all]
names_violent = [f.name for f in frames_violent]
names_property = [f.name for f in frames_property]
names_drug = [f.name for f in frames_drug]

# 14) Updatemenus for category selector and play/pause controls
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.0, y=1.08,
            xanchor="left", yanchor="top",
            buttons=[
                dict(label="All crimes", method="animate",
                     args=[names_all, {"mode": "immediate", "frame": {"duration": 500, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Violent crimes", method="animate",
                     args=[names_violent, {"mode": "immediate", "frame": {"duration": 500, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Property crimes", method="animate",
                     args=[names_property, {"mode": "immediate", "frame": {"duration": 500, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Drug-related", method="animate",
                     args=[names_drug, {"mode": "immediate", "frame": {"duration": 500, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}])
            ]
        )
    ]
)

# 15) Slider for All category by default (optional)
sliders = [dict(
    active=0,
    x=0.05, y=0.03,
    xanchor="left", yanchor="bottom",
    len=0.9,
    pad={"b": 10, "t": 10},
    currentvalue={"prefix": "Month: ", "visible": True},
    steps=[dict(
        label=ym.split('|')[-1],
        method="animate",
        args=[[f"ALL|{ym.split('|')[-1]}"],
              {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]
    ) for ym in names_all]
)]
fig.update_layout(sliders=sliders)

# 16) Save and/or show
out_path = "crime_hexmap.html"
fig.write_html(out_path, include_plotlyjs='cdn', auto_open=False)
fig.show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.3 MB/s  0:00:00


Output()

## 4. Spatiotemporal Clustering (DBSCAN)

In [ ]:
%%calliope ask gemini:gemini-2.5-pro --format code
Using crime_df:

Perform spatiotemporal clustering:
1. Combine spatial (lat, lon) and temporal (hour, day) features
2. Normalize features appropriately
3. Apply DBSCAN clustering
4. Identify crime hotspot clusters
5. For each cluster, calculate:
   - Geographic centroid
   - Time centroid (peak hour/day)
   - Size (number of crimes)
   - Dominant crime types
6. Visualize on map:
   - Color by cluster
   - Size by cluster density
   - Label with dominant crime type

Store cluster labels in crime_df.
Print top 10 hotspot clusters with characteristics.

## 5. Crime Type Relationship Network

In [ ]:
%%calliope ask anthropic-chat:claude-sonnet-4-5-20250929 --format code
Using crime_df:

Build network of crime type co-occurrences:
1. Group by location/time window to find co-occurring crimes
2. Create network where:
   - Nodes = crime types
   - Edges = frequency of co-occurrence within same area/time
3. Use networkx for graph construction
4. Calculate network metrics:
   - Node centrality (most connected crime types)
   - Community detection
   - Clustering coefficient
5. Visualize with plotly network graph:
   - Node size = frequency
   - Edge width = co-occurrence strength
   - Color by community
   - Interactive hover with stats

Print most connected crime types and communities found.

## 6. Animated Crime Wave Visualization

In [ ]:
%%calliope ask openai-chat:gpt-4o --format code
Using crime_df:

Create animated visualization of crime patterns:
1. Aggregate by date and location (grid cells)
2. Create plotly animated scatter_mapbox:
   - Animation frame = date (daily or weekly)
   - Marker size = crime count that day
   - Color = dominant crime type
   - Trail effect showing recent activity
3. Add play/pause controls
4. Show cumulative count ticker
5. Add slider to control animation speed
6. Highlight major crime spikes

Make it interactive and visually striking.

## 7. Seasonal Decomposition by Crime Type

In [ ]:
%%calliope ask cohere:command-r-plus-08-2024 --format code
Using crime_df:

Analyze seasonality for top 5 crime types:
1. Create daily time series for each crime type
2. For each, perform seasonal decomposition:
   - Trend
   - Seasonal (weekly and yearly)
   - Residual
3. Create subplot grid (5 rows x 4 cols):
   - Each row = one crime type
   - Cols = observed, trend, seasonal, residual
4. Calculate seasonal strength for each
5. Identify which crimes are most seasonal
6. Create summary heatmap: crime type x month showing avg count

Print seasonality rankings.

## 8. Predictive Hotspot Modeling

In [ ]:
%%calliope ask gemini:gemini-2.0-flash --format code
Using crime_df:

Build simple hotspot prediction:
1. Create grid cells over LA area
2. Calculate historical crime density per cell
3. Add temporal features: day_of_week, hour, month
4. Train simple model (logistic regression or random forest):
   - Input: cell location + time features
   - Output: high/low crime risk
5. Generate predictions for next week
6. Visualize predicted hotspots on map:
   - Color by risk level
   - Size by predicted intensity
7. Compare predictions with actual (if validation data available)

Print model accuracy metrics.

## 9. Chord Diagram: Crime Type Co-occurrence

In [ ]:
%%calliope ask mistralai:mistral-large-latest --format code
Using crime_df:

Create chord diagram showing crime relationships:
1. Calculate co-occurrence matrix for top 15 crime types
2. Use plotly or holoviews to create chord diagram
3. Chord thickness = co-occurrence frequency
4. Color-code crime categories:
   - Violent crimes
   - Property crimes
   - Drug-related
   - Other
5. Add hover showing specific counts
6. Make it interactive

Alternative: Use network circular layout if chord not available.

## 10. Comparative Dashboard: Weekday vs Weekend

In [ ]:
%%calliope ask anthropic-chat:claude-3-7-sonnet-20250219 --format code
Using crime_df:

Compare weekday vs weekend crime patterns:
1. Split data by is_weekend flag
2. Create 2x3 comparison dashboard:
   - Row 1 (Weekday):
     • Hourly distribution
     • Geographic heatmap
     • Crime type breakdown
   - Row 2 (Weekend): Same metrics
3. Add difference heatmap showing:
   - Where weekend crime > weekday
   - Statistical significance markers
4. Calculate and display:
   - Crime types with biggest weekend/weekday difference
   - Locations with different patterns

Export as 'weekday_vs_weekend_report.html'.

## 11. 3D Visualization: Space-Time Cube

In [ ]:
%%calliope ask gpto --format code
Using crime_df:

Create 3D space-time visualization:
1. Use plotly 3D scatter:
   - X-axis: Longitude
   - Y-axis: Latitude
   - Z-axis: Time (hours since start date)
2. Color by crime type
3. Size by severity if available
4. Sample to 10k points for performance
5. Add trajectory lines connecting related crimes
6. Make it rotatable and zoomable
7. Add time slicing slider

Create stunning 3D visualization showing spatiotemporal crime patterns.